## Agent workbench

The workbench is the layer above the notebook tools. It turns a user's
working taste into a task contract, compiles focused context, records a
baseline, and scores the resulting patch against small-diff gates.

It deliberately does not replace `execute_plan`. Instead, it prepares the
plan that the inner notebook agent should execute and checks whether the
result stayed small, direct, and aligned with the user's preferences.

### Production contract

The agent workbench is experimental until proven by focused tests. It may prepare context and budgets for small-diff work, but MCP exposure should stay optional unless it has stable output contracts, clear stop rules, and tests for execute and non-execute modes.

In [ ]:
#| default_exp workbench

In [ ]:
#| export
import ast
import json
import re

from contextlib import redirect_stderr, redirect_stdout
from copy import deepcopy
from io import StringIO
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.edit_interactive import execute_plan
from nbskill.foundation import (
    cap_text, cell_class_names, cell_source, file_hash, file_line_count,
    generated_owner, git_diff_stats, git_root, git_status_paths,
    git_tracked_paths, notebook_paths, source_without_directives,
)
from nbskill.graph import notebook_order_problems, placement_advice, reuse_advice, symbol_usage_summary
from nbskill.read import file_context
from nbskill.review import notebook_validation_problems, style_report

### Taste and task contracts

Taste is a compact, durable description of how the user wants work done.
The project profile is versioned in `.nbskill/taste.json`; an optional user
profile can override it. The task contract can override both for one run.

In [ ]:
#| export
BUILTIN_TASTE_PROFILE = {
    "version": 1,
    "preferences": [
        "Prefer the smallest coherent diff and ask only when ambiguity materially changes it.",
        "Use nbskill MCP tools for normal notebook reads, edits, execution, review, and diagnostics.",
        "Write notebooks as a story: rationale, implementation, visible example, focused test.",
        "Keep cells small, explain why the shape exists, and cross-reference downstream uses.",
    ],
    "aversions": [
        "Backwards-compatibility scaffolding unless requested.",
        "Generated-file edits without notebook source changes.",
        "Raw notebook JSON edits, large cells, missing docs, hidden outputs, and speculative architecture.",
    ],
    "defaults": {
        "compatibility": "not_required",
        "public_api": "forbidden",
        "gate_style": "soft",
    },
}

In [ ]:
#| export
DEFAULT_BUDGETS = {
    "max_files": 2,
    "max_cells": 3,
    "max_added_lines": 80,
    "max_context_notebooks": 20,
    "max_context_chars": 50000,
    "max_rendered_plan_chars": 50000,
    "max_agent_steps": 20,
    "max_agent_timeout": 120,
    "max_run_secs": 30,
    "max_context_commands": 8,
    "context_steps": 2,
}

_WORKBENCH_MAX_NOTEBOOKS = 20
_WORKBENCH_MAX_CONTEXT_CHARS = 50000
_WORKBENCH_MAX_RENDERED_PLAN_CHARS = 50000

In [ ]:
#| export
DEFAULT_VERIFICATION = [
    "Run the narrowest notebook execution possible, using check_only=True when available.",
    "Review code-cell changes with diff_nb.",
    "Run doctor(scopes='error,warning,style') or style_check on touched notebooks.",
]

In [ ]:
#| export
def _deep_merge(base, override):
    result = deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = deepcopy(value)
    return result

In [ ]:
#| export
def _read_json(path):
    path = Path(path).expanduser()
    if not path.exists(): return {}
    data = json.loads(path.read_text(encoding="utf-8"))
    if data is None: return {}
    if not isinstance(data, dict): raise ValueError(f"{path} must contain a JSON object")
    return data

In [ ]:
#| export
def _project_taste_path(project_path):
    path = Path(project_path).expanduser()
    if path.is_file(): path = path.parent
    for candidate in [path, *path.parents]:
        taste = candidate / ".nbskill" / "taste.json"
        if taste.exists(): return taste
    return path / ".nbskill" / "taste.json"

In [ ]:
#| export
def load_taste_profile(project_path=".", user_path="~/.nbskill/taste.json") -> dict:
    "Load built-in, project, and optional user taste profiles with later profiles winning."
    taste = deepcopy(BUILTIN_TASTE_PROFILE)
    taste = _deep_merge(taste, _read_json(_project_taste_path(project_path)))
    if user_path: taste = _deep_merge(taste, _read_json(user_path))
    return taste

In [ ]:
#| export
def _contract_overrides(overrides):
    overrides = dict(overrides)
    budget_keys = set(DEFAULT_BUDGETS)
    budgets = {key: overrides.pop(key) for key in list(overrides) if key in budget_keys}
    if budgets: overrides["budgets"] = _deep_merge(overrides.get("budgets", {}), budgets)
    return overrides

In [ ]:
#| export
def make_task_contract(goal, **overrides) -> dict:
    "Create a concrete task contract with conservative small-diff defaults."
    contract = {
        "goal": goal,
        "non_goals": [],
        "budgets": deepcopy(DEFAULT_BUDGETS),
        "compatibility": "not_required",
        "public_api": "forbidden",
        "edit_policy": {
            "prefer_existing_code": True,
            "forbid_generated_only_edits": True,
            "stop_when_smallest_valid_diff_passes": True,
        },
        "verification": list(DEFAULT_VERIFICATION),
        "taste": {},
    }
    return _deep_merge(contract, _contract_overrides(overrides))

### Repository and notebook state

The scorer compares a baseline with the current state so it can ignore
pre-existing dirt and focus on what changed during the workbench run.

In [ ]:
#| export
def _public_symbols_in_cell(cell):
    if "exported_code" not in cell_class_names(cell): return []
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    return [
        node.name for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and not node.name.startswith("_")
    ]

In [ ]:
#| export
def _notebook_cell_sources(path):
    try: nb = read_nb(path)
    except FileNotFoundError: return {}
    return {getattr(cell, "id", f"idx-{idx}"): cell_source(cell) for idx, cell in enumerate(nb.cells)}

In [ ]:
#| export
def _notebook_public_symbols(path):
    try: nb = read_nb(path)
    except FileNotFoundError: return []
    symbols = []
    for cell in nb.cells:
        symbols.extend(_public_symbols_in_cell(cell))
    return sorted(set(symbols))

In [ ]:
#| export
def _generated_pairs(root):
    pairs = []
    for path in Path(root).rglob("*.py"):
        if any(part in {".git", ".venv", "__pycache__"} for part in path.parts): continue
        owner = generated_owner(path)
        if owner is not None: pairs.append((path, owner))
    return pairs

In [ ]:
#| export
def _style_summary(path):
    try: report = style_report(path, max_output_chars=4000, max_diagnostics=100)
    except BaseException as exc: return {"error": f"{type(exc).__name__}: {exc}"}
    return {
        "summary": report.get("summary", {}),
        "problem_chart": report.get("problem_chart", {}),
    }

In [ ]:
#| export
def capture_state(path=".") -> dict:
    "Capture git, notebook, export, and style state for later patch scoring."
    root = git_root(path)
    base = root or Path(path).expanduser()
    target = Path(path).expanduser()

    def _scope_rel():
        if str(path) in {"", "."}: return None
        try:
            rel = target.resolve().relative_to(base.resolve()).as_posix()
        except ValueError:
            return None
        return None if rel in {"", "."} else rel

    scope = _scope_rel()

    def _in_scope(rel):
        if scope is None: return True
        rel = str(rel)
        return rel == scope or rel.startswith(f"{scope.rstrip('/')}/")

    changed = {item for item in (git_status_paths(root) if root else set()) if _in_scope(item)}
    tracked = {item for item in (git_tracked_paths(root) if root else set()) if _in_scope(item)}
    stats = {key: value for key, value in (git_diff_stats(root) if root else {}).items() if _in_scope(key)}
    notebooks = notebook_paths(path if Path(path).exists() else base)
    rel_notebooks = []
    for nb_path in notebooks:
        try: rel_notebooks.append(nb_path.resolve().relative_to(base.resolve()).as_posix())
        except ValueError: rel_notebooks.append(str(nb_path))
    hash_paths = set(tracked) | changed | set(rel_notebooks)
    file_hashes = {
        rel: file_hash(base / rel)
        for rel in sorted(hash_paths)
        if (base / rel).exists() and (base / rel).is_file()
    }
    for rel, item in list(stats.items()):
        if item["added"] == 0 and rel not in tracked and (base / rel).exists():
            item["added"] = file_line_count(base / rel)
    generated_pairs = [
        {"path": str(py), "owner": str(owner)}
        for py, owner in _generated_pairs(base)
        if _in_scope(py) or _in_scope(owner)
    ]
    return {
        "root": str(base),
        "changed_paths": sorted(changed),
        "diff_stats": stats,
        "file_hashes": file_hashes,
        "notebook_cell_sources": {str(nb): _notebook_cell_sources(nb) for nb in notebooks},
        "public_symbols": {str(nb): _notebook_public_symbols(nb) for nb in notebooks},
        "validation_problem_count": len(notebook_validation_problems(path)),
        "order_problem_count": len(notebook_order_problems(path)),
        "style": _style_summary(path),
        "generated_pairs": generated_pairs,
    }

### Context compiler

The context compiler returns evidence cards, not a transcript. It keeps
stable cell ids visible so the executor can make targeted edits.

In [ ]:
#| export
def _tokens(text):
    stop = {"the", "and", "for", "with", "that", "this", "into", "from", "should", "agent"}
    return {item for item in re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", str(text).lower()) if item not in stop}

In [ ]:
#| export
def _score_text(text, tokens):
    low = str(text).lower()
    return sum(low.count(token) for token in tokens)

In [ ]:
#| export
def _call_text(func, **kwargs):
    out, err = StringIO(), StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        result = func(**kwargs)
    if isinstance(result, str) and result.strip(): return result.strip()
    return "\n".join(chunk.strip() for chunk in [out.getvalue(), err.getvalue()] if chunk.strip())

In [ ]:
#| export
def _cell_records(nb_path, tokens, limit=4):
    nb = read_nb(nb_path)
    records = []
    for idx, cell in enumerate(nb.cells):
        source = cell_source(cell)
        score = _score_text(source, tokens)
        classes = list(cell_class_names(cell))
        if score or "test_cell" in classes or "example_cell" in classes:
            records.append({
                "path": str(nb_path),
                "idx": idx,
                "id": getattr(cell, "id", ""),
                "first_line": source.strip().splitlines()[0] if source.strip() else "",
                "score": score,
            })
    return sorted(records, key=lambda item: (-item["score"], item["idx"]))[:limit]

In [ ]:
#| export
def _symbol_candidates(nb_path, tokens):
    names = []
    try: nb = read_nb(nb_path)
    except FileNotFoundError: return []
    for cell in nb.cells:
        for name in _public_symbols_in_cell(cell):
            if name.lower() in tokens or _score_text(name, tokens): names.append(name)
    return sorted(set(names))[:5]

In [ ]:
#| export
def compile_context(goal, path="nbs", contract=None, taste=None) -> dict:
    "Compile a focused context pack for an agent workbench run."
    contract = contract or make_task_contract(goal)
    taste = taste or load_taste_profile(project_path=path)
    budgets = contract.get("budgets", {})
    max_notebooks = min(int(budgets.get("max_context_notebooks", _WORKBENCH_MAX_NOTEBOOKS)), _WORKBENCH_MAX_NOTEBOOKS)
    max_context_chars = min(int(budgets.get("max_context_chars", _WORKBENCH_MAX_CONTEXT_CHARS)), _WORKBENCH_MAX_CONTEXT_CHARS)
    tokens = _tokens(goal)
    notebooks = notebook_paths(path)
    if len(notebooks) > max_notebooks:
        raise ValueError(
            f"agent_workbench context budget exceeded: {len(notebooks)} notebooks exceeds limit {max_notebooks}; "
            "pass a narrower notebook/path."
        )
    overview_cards = []
    for nb_path in notebooks:
        overview = _call_text(file_context, path=str(nb_path), verbose=True)
        overview_cards.append({
            "path": str(nb_path),
            "score": _score_text(overview, tokens),
            "overview": cap_text(overview, 1800),
        })
    selected = sorted(overview_cards, key=lambda item: (-item["score"], item["path"]))[:3]
    if not any(item["score"] for item in selected): selected = overview_cards[:3]
    evidence = []
    symbols = []
    for card in selected:
        nb_path = Path(card["path"])
        nb = read_nb(nb_path)
        cells_by_id = {getattr(cell, "id", ""): cell for cell in nb.cells}
        cell_cards = _cell_records(nb_path, tokens)
        for cell in cell_cards[:2]:
            target = cells_by_id.get(cell["id"])
            context = cell_source(target) if target is not None else ""
            evidence.append({**cell, "context": cap_text(context, 2200)})
        symbols.extend(_symbol_candidates(nb_path, tokens))
    symbol_summary = symbol_usage_summary(path, symbols) if symbols else ""
    reuse = reuse_advice(goal, path=path, top_k=5)
    placement = placement_advice(path=path, source=goal, top_k=5)
    state = capture_state(path)
    context = {
        "goal": goal,
        "contract": contract,
        "taste": taste,
        "selected_notebooks": selected,
        "evidence": evidence,
        "symbols": symbols,
        "symbol_summary": cap_text(symbol_summary, 2000),
        "reuse_advice": reuse,
        "placement_advice": placement,
        "doctor_warnings": {
            "validation_problem_count": state["validation_problem_count"],
            "order_problem_count": state["order_problem_count"],
        },
        "style_summary": state["style"].get("summary", {}),
        "verification_hints": contract.get("verification", []),
    }
    serialized = json.dumps(context, default=str)
    if len(serialized) > max_context_chars:
        context["reuse_advice"] = {"summary": "omitted: context budget reached"}
        context["placement_advice"] = {"summary": "omitted: context budget reached"}
        context["evidence"] = context["evidence"][:3]
    return context

### Patch scoring and gates

Hard gates protect scope and stale context. Style and readability are
reported as deltas so a task can improve the touched area without needing
the whole repository to be clean first.

In [ ]:
#| export
def _changed_files(baseline, current):
    before = baseline.get("file_hashes", {})
    after = current.get("file_hashes", {})
    paths = set(before) | set(after)
    return sorted(path for path in paths if before.get(path) != after.get(path))

In [ ]:
#| export
def _changed_cells(baseline, current):
    changed = []
    before_all = baseline.get("notebook_cell_sources", {})
    after_all = current.get("notebook_cell_sources", {})
    for nb_path in sorted(set(before_all) | set(after_all)):
        before = before_all.get(nb_path, {})
        after = after_all.get(nb_path, {})
        for cell_id in sorted(set(before) | set(after)):
            if before.get(cell_id) != after.get(cell_id):
                changed.append({"path": nb_path, "cell_id": cell_id})
    return changed

In [ ]:
#| export
def _added_line_delta(baseline, current):
    before = baseline.get("diff_stats", {})
    after = current.get("diff_stats", {})
    total = 0
    for path in set(before) | set(after):
        total += max(0, after.get(path, {}).get("added", 0) - before.get(path, {}).get("added", 0))
    return total

In [ ]:
#| export
def _added_public_symbols(baseline, current):
    added = []
    before_all = baseline.get("public_symbols", {})
    after_all = current.get("public_symbols", {})
    for nb_path, symbols in after_all.items():
        for symbol in sorted(set(symbols) - set(before_all.get(nb_path, []))):
            added.append({"path": nb_path, "symbol": symbol})
    return added

In [ ]:
#| export
def _export_pair_failures(current, touched_files):
    touched = {str(Path(path)) for path in touched_files}
    failures = []
    for pair in current.get("generated_pairs", []):
        py_path = str(Path(pair["path"]))
        owner = str(Path(pair["owner"]))
        py_touched = any(path == py_path or Path(current["root"], path).resolve() == Path(py_path).resolve() for path in touched)
        owner_touched = any(path == owner or Path(current["root"], path).resolve() == Path(owner).resolve() for path in touched)
        if py_touched and not owner_touched:
            failures.append({"code": "generated_without_notebook", "path": py_path, "owner": owner})
        if owner_touched and not py_touched:
            failures.append({"code": "notebook_export_missing", "path": owner, "generated": py_path})
    return failures

In [ ]:
#| export
def _style_delta(baseline, current):
    before = baseline.get("style", {}).get("summary", {})
    after = current.get("style", {}).get("summary", {})
    keys = sorted(set(before) | set(after))
    return {
        key: after.get(key, 0) - before.get(key, 0)
        for key in keys
        if isinstance(after.get(key, 0), int) and isinstance(before.get(key, 0), int)
    }

In [ ]:
#| export
def score_patch(baseline, contract, path=".") -> dict:
    "Score current repository state against a baseline and task contract."
    current = capture_state(path)
    budgets = contract.get("budgets", {})
    touched_files = _changed_files(baseline, current)
    changed_cells = _changed_cells(baseline, current)
    added_lines = _added_line_delta(baseline, current)
    added_public = _added_public_symbols(baseline, current)
    hard_failures = []
    if len(touched_files) > budgets.get("max_files", DEFAULT_BUDGETS["max_files"]):
        hard_failures.append({"code": "max_files", "limit": budgets.get("max_files"), "actual": len(touched_files)})
    if len(changed_cells) > budgets.get("max_cells", DEFAULT_BUDGETS["max_cells"]):
        hard_failures.append({"code": "max_cells", "limit": budgets.get("max_cells"), "actual": len(changed_cells)})
    if added_lines > budgets.get("max_added_lines", DEFAULT_BUDGETS["max_added_lines"]):
        hard_failures.append({"code": "max_added_lines", "limit": budgets.get("max_added_lines"), "actual": added_lines})
    if contract.get("public_api") == "forbidden" and added_public:
        hard_failures.append({"code": "public_api_added", "symbols": added_public})
    hard_failures.extend(_export_pair_failures(current, touched_files))
    if current["validation_problem_count"] > baseline.get("validation_problem_count", 0):
        hard_failures.append({"code": "validation_errors_added"})
    if current["order_problem_count"] > baseline.get("order_problem_count", 0):
        hard_failures.append({"code": "order_errors_added"})
    return {
        "passed": not hard_failures,
        "hard_failures": hard_failures,
        "touched_files": touched_files,
        "changed_cells": changed_cells,
        "added_lines": added_lines,
        "added_public_symbols": added_public,
        "style_delta": _style_delta(baseline, current),
        "baseline": baseline,
        "current": current,
    }

### Workbench entrypoint

`agent_workbench` is safe by default. With `execute=False` it only returns
the contract, context, baseline, and rendered plan. With `execute=True`, it
requires one notebook and delegates the bounded edit to `execute_plan`.

In [ ]:
#| export
def _load_contract_file(contract_file):
    if not contract_file: return {}
    return _read_json(contract_file)

In [ ]:
#| export
def _render_list(items):
    return "\n".join(f"- {item}" for item in items) if items else "- (none)"

In [ ]:
#| export
def _render_budget_lines(contract):
    budgets = contract.get("budgets", {})
    lines = ["Budgets:"]
    lines.extend(f"- {key}={budgets.get(key)}" for key in DEFAULT_BUDGETS)
    lines.extend(
        [
            f"- compatibility={contract.get('compatibility')}",
            f"- public_api={contract.get('public_api')}",
        ]
    )
    return lines

In [ ]:
#| export
def _render_workbench_header(contract, taste):
    return [
        "Agent workbench task",
        "",
        f"Goal: {contract['goal']}",
        "",
        "Taste preferences:",
        _render_list(taste.get("preferences", [])),
        "",
        "Aversions:",
        _render_list(taste.get("aversions", [])),
        "",
        *_render_budget_lines(contract),
    ]

In [ ]:
#| export
def _render_context_lines(context):
    lines = ["", "Relevant notebooks:"]
    lines.extend(
        f"- {item['path']} score={item['score']}"
        for item in context.get("selected_notebooks", [])
    )
    lines.extend(["", "Evidence cells:"])
    lines.extend(f"- {item['path']} id={item['id']}" for item in context.get("evidence", []))
    reuse = context.get("reuse_advice") or {}
    if reuse.get("matches") or reuse.get("notebooks"):
        lines.extend(["", "Search-before-write:"])
        for item in reuse.get("matches", [])[:3]:
            lines.append(f"- reuse {item['symbol']} in {item['path']} id={item['cell_id']} score={item['score']}")
        for item in reuse.get("notebooks", [])[:3]:
            lines.append(f"- inspect {item['path']} score={item['score']}")
    placement = context.get("placement_advice") or {}
    if placement.get("candidates"):
        lines.extend(["", "Placement candidates:"])
        for item in placement.get("candidates", [])[:3]:
            lines.append(f"- {item['path']} score={item['score']} reasons={'; '.join(item.get('reasons', []))}")
    if context.get("symbol_summary"):
        lines.extend(["", "Symbol consequences:", context["symbol_summary"]])
    return lines

In [ ]:
#| export
def _render_notebook_craft_lines():
    body = """
Notebook craft:
- Keep notebooks coherent: small markdown and code cells, one idea at a time.
- Add descriptions before code, examples with visible outputs, and small tests near the behavior they protect.
- Explain rationale: why the shape solves the problem, what tradeoffs it rejects, and why nearby alternatives do not fit.
- Add cross-references for exported symbols: where the code is used later, and why that reuse matters.
- End with diff_nb plus doctor(scopes='error,warning,style') or style_check, and resolve new notebook smells before stopping.
"""
    return ["", *body.strip().splitlines()]

In [ ]:
#| export
def _render_workbench_plan(contract, taste, context):
    "Render the executor-facing plan with taste, budgets, context, and stop rules."
    lines = [
        *_render_workbench_header(contract, taste),
        *_render_context_lines(context),
        *_render_notebook_craft_lines(),
        "",
        "Stop rules:",
        "- Prefer the smallest valid diff over a broad complete rewrite.",
        "- Modify existing code before adding helpers.",
        "- Do not add compatibility scaffolding unless the contract says so.",
        "- Stop once the contract is satisfied and verification is green.",
        "",
        "Verification:",
        _render_list(contract.get("verification", [])),
    ]
    return "\n".join(lines).rstrip()

In [ ]:
#| export
def agent_workbench_result(
    goal: str,
    notebook: str | None = None,
    contract_file: str | None = None,
    execute: bool = False,
    max_steps: int = 8,
    timeout: int = 30,
) -> dict:
    "Build the structured workbench result without CLI printing side effects."
    overrides = _load_contract_file(contract_file)
    contract_goal = overrides.pop("goal", goal)
    contract = make_task_contract(contract_goal, **overrides)
    context_path = notebook or "nbs"
    taste = _deep_merge(
        load_taste_profile(project_path=context_path), contract.get("taste", {})
    )
    context = compile_context(
        contract["goal"], path=context_path, contract=contract, taste=taste
    )
    baseline = capture_state(context_path)
    budgets = contract.get("budgets", {})
    max_plan_chars = min(
        int(
            budgets.get(
                "max_rendered_plan_chars", _WORKBENCH_MAX_RENDERED_PLAN_CHARS
            )
        ),
        _WORKBENCH_MAX_RENDERED_PLAN_CHARS,
    )
    rendered_plan = cap_text(_render_workbench_plan(contract, taste, context), max_plan_chars)
    result = {
        "summary": "agent_workbench prepared execution context",
        "contract": contract,
        "taste": taste,
        "context": context,
        "baseline": baseline,
        "rendered_plan": rendered_plan,
        "expected_gates": {
            "hard": [
                "max_files",
                "max_cells",
                "max_added_lines",
                "public_api",
                "generated/source sync",
                "new doctor errors",
            ],
            "soft": ["style_delta", "readability_delta", "test_clarity"],
        },
    }
    if execute:
        if not notebook:
            raise ValueError("agent_workbench execute=True requires notebook")
        max_steps = min(
            int(max_steps),
            int(budgets.get("max_agent_steps", DEFAULT_BUDGETS["max_agent_steps"])),
        )
        timeout = min(
            int(timeout),
            int(budgets.get("max_run_secs", DEFAULT_BUDGETS["max_run_secs"])),
        )
        max_context_commands = int(
            budgets.get(
                "max_context_commands", DEFAULT_BUDGETS["max_context_commands"]
            )
        )
        context_steps = int(
            budgets.get("context_steps", DEFAULT_BUDGETS["context_steps"])
        )
        execution = execute_plan(
            notebook=notebook,
            plan=rendered_plan,
            max_steps=max_steps,
            timeout=timeout,
            max_context_commands=max_context_commands,
            context_steps=context_steps,
        )
        result["execution"] = execution
        result["score"] = score_patch(baseline, contract, path=context_path)
        result["summary"] = "agent_workbench executed plan"
    return result

In [ ]:
#| export
def agent_workbench(
    goal: str,  # Desired software-development outcome
    notebook: str | None = None,  # Required when execute=True; also narrows context when provided
    contract_file: str | None = None,  # Optional JSON contract overrides
    execute: bool = False,  # Execute the rendered plan through execute_plan
    max_steps: int = 8,  # Maximum inner-agent steps when executing
    timeout: int = 30,  # Per-cell timeout for execute_plan
) -> dict:
    "Prepare or execute a taste-aware, small-diff agent workbench run."
    return agent_workbench_result(
        goal, notebook=notebook, contract_file=contract_file, execute=execute,
        max_steps=max_steps, timeout=timeout,
    )

### Tests

These tests use tiny temporary notebooks and monkeypatch the inner executor
so the workbench behavior stays deterministic.

In [ ]:
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_test_nb

from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
#| hide
from contextlib import contextmanager


@contextmanager
def _demo_root(name):
    root = demo_path(name)
    root.mkdir()
    try:
        yield root
    finally:
        remove_demo_path(root)


@contextmanager
def _patched_execute_plan(fake):
    global execute_plan
    original = execute_plan
    execute_plan = fake
    try:
        yield
    finally:
        execute_plan = original

In [ ]:
with _demo_root("11_workbench_taste") as root:
    user_path = root / "user_taste.json"
    (root / ".nbskill").mkdir(parents=True)
    (root / ".nbskill" / "taste.json").write_text(json.dumps({
        "preferences": ["project preference"],
        "defaults": {"compatibility": "project"},
    }), encoding="utf-8")
    user_path.write_text(json.dumps({
        "preferences": ["user preference"],
        "defaults": {"public_api": "user"},
    }), encoding="utf-8")
    taste = load_taste_profile(root, user_path)

assert taste["preferences"] == ["user preference"]
assert taste["defaults"]["compatibility"] == "project"
assert taste["defaults"]["public_api"] == "user"

In [ ]:
#| hide
contract = make_task_contract(
    "Add focused behavior", max_files=1, taste={"preferences": ["task preference"]}
)
assert contract["budgets"]["max_files"] == 1
assert contract["budgets"]["max_cells"] == 3
assert contract["budgets"]["max_run_secs"] == 30
assert contract["budgets"]["max_context_commands"] == 8
assert contract["budgets"]["context_steps"] == 2
assert contract["compatibility"] == "not_required"
assert contract["public_api"] == "forbidden"
assert contract["taste"]["preferences"] == ["task preference"]

In [ ]:
#| hide
contract = make_task_contract("render plan")
context = {
    "selected_notebooks": [{"path": "nbs/demo.ipynb", "score": 1}],
    "evidence": [{"path": "nbs/demo.ipynb", "id": "abc", "reason": "matches goal"}],
    "symbols": ["demo"],
    "symbol_summary": "demo usage",
    "reuse_advice": {
        "matches": [
            {"symbol": "demo", "path": "nbs/demo.ipynb", "cell_id": "abc", "score": 12}
        ],
        "notebooks": [{"path": "nbs/demo.ipynb", "score": 8}],
    },
    "placement_advice": {
        "candidates": [{"path": "nbs/demo.ipynb", "score": 8, "reasons": ["demo"]}]
    },
    "doctor_warnings": {"validation_problem_count": 0, "order_problem_count": 0},
    "verification_hints": ["run focused test"],
}
plan = _render_workbench_plan(contract, BUILTIN_TASTE_PROFILE, context)
assert "Agent workbench task" in plan
assert "Taste preferences" in plan
assert "Notebook craft" in plan
assert "Search-before-write" in plan
assert "Placement candidates" in plan
assert "doctor(scopes='error,warning,style')" in plan
assert "max_run_secs=30" in plan
assert "max_context_commands=8" in plan
assert "context_steps=2" in plan
assert "nbs/demo.ipynb" in plan

In [ ]:
with _demo_root("11_workbench_json") as root:
    contract = root / "contract.json"
    contract.write_text("null", encoding="utf-8")
    loaded_contract = _load_contract_file(contract)

    contract.write_text("[]", encoding="utf-8")
    try:
        _load_contract_file(contract)
    except ValueError as exc:
        message = str(exc)
    else:
        raise AssertionError("list contract should fail")

assert loaded_contract == {}
assert "JSON object" in message

In [ ]:
with _demo_root("11_workbench_context") as root:
    nb_path = root / "demo.ipynb"
    _write_test_nb(new_nb([
        mk_cell("## Demo\nSmall docs for parser work.", cell_type="markdown"),
        mk_cell("#| export\ndef parse_value(text):\n    return text.strip()"),
        mk_cell("assert parse_value(' x ') == 'x'"),
    ]), nb_path)
    pack = compile_context("improve parse_value behavior", path=str(root))

assert pack["selected_notebooks"]
assert pack["evidence"]
assert pack["evidence"][0]["id"]
assert "parse_value" in pack["symbol_summary"]
assert pack["reuse_advice"]["matches"]
assert pack["placement_advice"]["candidates"]

In [ ]:
#| hide
out = StringIO()

with _demo_root("11_workbench_prepare") as root:
    nb_path = root / "demo.ipynb"
    _write_test_nb(
        new_nb(
            [
                mk_cell(
                    "## Demo notebook\nThis tiny notebook gives nbskill tools something real to inspect.",
                    cell_type="markdown",
                ),
                mk_cell("#| export\ndef demo_answer():\n    return 42"),
                mk_cell("assert demo_answer() == 42"),
            ]
        ),
        nb_path,
    )
    before = [cell_source(cell) for cell in read_nb(nb_path).cells]
    prepared = agent_workbench_result(
        "make demo_answer direct", notebook=str(nb_path.resolve()), execute=False
    )
    with redirect_stdout(out):
        result = agent_workbench(
            "make demo_answer direct", notebook=str(nb_path.resolve()), execute=False
        )
    after = [cell_source(cell) for cell in read_nb(nb_path).cells]

assert before == after
assert result["summary"] == "agent_workbench prepared execution context"
assert out.getvalue() == ""
assert prepared["contract"]["budgets"]["max_added_lines"] == 80
assert prepared["contract"]["budgets"]["max_run_secs"] == 30
assert prepared["contract"]["budgets"]["max_context_commands"] == 8
assert prepared["contract"]["budgets"]["context_steps"] == 2
assert "Agent workbench task" in prepared["rendered_plan"]
assert "max_run_secs=30" in prepared["rendered_plan"]
assert "max_context_commands=8" in prepared["rendered_plan"]
assert "context_steps=2" in prepared["rendered_plan"]

In [ ]:
#| hide
calls = []


def fake_execute_plan(**kwargs):
    calls.append(kwargs)
    return {"summary": "fake execution", "history": [], "diff": ""}


with write_demo_notebook("11_workbench_execute.ipynb") as nb_path:
    with _patched_execute_plan(fake_execute_plan):
        result = agent_workbench_result(
            "touch nothing",
            notebook=str(nb_path.resolve()),
            execute=True,
            max_steps=99,
            timeout=99,
        )

assert calls and calls[0]["notebook"] == str(nb_path.resolve())
assert "Taste preferences" in calls[0]["plan"]
assert calls[0]["max_steps"] == DEFAULT_BUDGETS["max_agent_steps"]
assert calls[0]["timeout"] == DEFAULT_BUDGETS["max_run_secs"]
assert calls[0]["max_context_commands"] == DEFAULT_BUDGETS["max_context_commands"]
assert calls[0]["context_steps"] == DEFAULT_BUDGETS["context_steps"]
assert result["execution"]["summary"] == "fake execution"
assert result["score"]["passed"]

In [ ]:
with _demo_root("11_workbench_score") as root:
    nb_path = root / "score.ipynb"
    _write_test_nb(new_nb([
        mk_cell("## Score", cell_type="markdown"),
        mk_cell("#| export\ndef existing():\n    return 1"),
    ]), nb_path)
    baseline = capture_state(str(root.resolve()))
    _write_test_nb(new_nb([
        mk_cell("## Score", cell_type="markdown"),
        mk_cell("#| export\ndef existing():\n    return 1"),
        mk_cell("#| export\ndef new_api():\n    return 2"),
    ]), nb_path)
    contract = make_task_contract("small edit", max_cells=1, public_api="forbidden")
    score = score_patch(baseline, contract, path=str(root.resolve()))

assert not score["passed"]
assert any(item["code"] == "public_api_added" for item in score["hard_failures"])
assert score["changed_cells"]